# Költségigény elemzés

Ez a jegyzetfüzet bemutatja, hogyan lehet olyan ügynököket létrehozni, amelyek bővítményeket használnak a helyi számlaképekből származó utazási költségek feldolgozására, költségigény e-mail generálására, valamint a költségadatok kördiagramon való megjelenítésére. Az ügynökök dinamikusan választanak függvényeket a feladat kontextusa alapján.

Lépések:
1. Az OCR ügynök feldolgozza a helyi számlaképet és kinyeri az utazási költségadatokat.
2. Az Email ügynök generál egy költségigény e-mailt.

### Egy utazási költséghelyzet példája:
Képzelj el egy dolgozót, aki üzleti megbeszélés miatt utazik egy másik városba. A céged szabályzata szerint minden ésszerű, utazással kapcsolatos költséget megtérítenek. Íme egy bontás a lehetséges utazási költségekről:
- Közlekedés:
Oda-vissza repülőjegy az otthoni városodból a célvárosba.
Taxi vagy fuvarszolgáltatás az repülőtérre és onnan vissza.
Helyi közlekedés a célvárosban (például tömegközlekedés, bérautó vagy taxi).

- Szállás:
Három éjszakás szállás egy középkategóriás üzleti szállodában, amely közel van a találkozó helyszínéhez.

- Étkezés:
Napi étkezési díj reggelire, ebédre és vacsorára, a cég napidíj szabályzata alapján.

- Egyéb költségek:
Parkolási díjak a repülőtéren.
Internethasználati díjak a szállodában.
Borravalók vagy kis kiszolgálási díjak.

- Dokumentáció:
Minden számlát (repülőjárat, taxi, szálloda, étkezés stb.) és egy kitöltött költségjelentést benyújtasz a megtérítéshez.


## Szükséges könyvtárak importálása

Importáld a jegyzetfüzethez szükséges könyvtárakat és modulokat.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Kiadási modellek definiálása

 Hozzon létre egy Pydantic modellt az egyedi kiadásokhoz és egy ExpenseFormatter osztályt, amely átalakítja a felhasználói lekérdezést strukturált kiadási adatokká.

 Minden kiadás a következő formátumban lesz ábrázolva:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Az e-mail generálása

Hozz létre egy eszköz függvényt, amely egy költségtérítés benyújtásához szükséges e-mailt generál.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Kiszámolja a költségek összegét, és az adatokat egy e-mail törzsbe formázza.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Utazási költségek kinyerésére szolgáló eszköz a blokk képeiből

Hozzon létre egy eszközfüggvényt az utazási költségek kinyerésére a blokk képeiből.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Beolvassa a blokk képét, base64-ben kódolja, és visszaadja az adat URI-t az elemzéshez az agent számára.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Költségek feldolgozása

Határozza meg az ügynököket, és kösse őket egy szekvenciális munkafolyamatba a `WorkflowBuilder` segítségével.
- Az OCR ügynök a `load_receipt_image` eszközt használva strukturált költségadatokat von ki a blokkon lévő képből.
- Az Email ügynök a kinyert adatokat használva a `generate_expense_email` eszközzel professzionális költségtérítési e-mailt generál.
- A `WorkflowBuilder` az `add_edge` segítségével szekvenciális csővezetéket hoz létre: OCR ügynök → Email ügynök.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Fő függvény

Építse fel a szekvenciális munkafolyamatot, és futtassa azt a blokk képfeldolgozásához és a költségtérítési e-mail létrehozásához.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ezt a dokumentumot az AI fordító szolgáltatásával, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár igyekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti dokumentum saját nyelvén tekintendő hiteles forrásnak. Kritikus információk esetén javasolt szakmai, emberi fordítást igénybe venni. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
